# RVC — Train trên Google Colab / Kaggle (một notebook)

Notebook này **hướng dẫn chi tiết** và **tự tạo** package `training_pipeline`, file patch, script tải weight — bạn **chỉ cần chạy lần lượt các ô** từ trên xuống.

## Vì sao không nhét hết `infer/` vào một file `.ipynb`?

Thư mục `infer/` + `configs/` của RVC có **hàng nghìn dòng** (mô hình, train loop, UVR…). Nhúng nguyên văn vào notebook sẽ **vượt giới hạn thực tế**, khó đọc và **không đồng bộ** khi RVC cập nhật.

**Cách làm chuẩn trên Colab:** tải **mã RVC upstream một lần** (`git clone`), sau đó notebook **ghi đè / bổ sung** phần điều phối (`training_pipeline`) và **các patch** (PyTorch 2.6+ / Matplotlib / subprocess path) đã được kiểm chứng.

## Luồng tổng quát

1. Cấu hình `RVC_ROOT`, URL clone (mặc định: repo RVC chính thức).
2. Cài PyTorch (GPU), `fairseq`, phụ thuộc.
3. Clone RVC → có `infer/`, `configs/`, `i18n/`.
4. Sinh `training_pipeline/` + `infer/lib/fairseq_torch_load_compat.py` + vá các file infer (từ nội dung đóng gói trong notebook).
5. Tải `assets/` (Hubert, pretrained…).
6. Chuẩn bị `logs/mute` (tải từ nhánh repo hoặc tự tạo tối thiểu).
7. Upload / đặt `.wav` vào `datasets/`.
8. Chạy preprocess → F0+Hubert → train → index.

**Lưu ý:** Colab miễn phí có giới hạn phiên; train dài nên dùng Colab Pro hoặc Kaggle GPU.


## 0) Cấu hình (sửa cho phù hợp)

Ô dưới đặt biến toàn cục: thư mục làm việc, URL Git, có dùng mirror Hugging Face hay không.


In [ ]:
# --- CẤU HÌNH ---
import os
from pathlib import Path

# Thư mục gốc chứa infer/, configs/ sau khi clone (Linux Colab/Kaggle thường dùng /content)
RVC_ROOT = Path(os.environ.get("RVC_ROOT", "/content/Retrieval-based-Voice-Conversion-WebUI"))

# Repo RVC chính thức (đủ infer + configs). Nếu bạn fork và thêm sửa, đổi URL tại đây.
GIT_REPO = os.environ.get(
    "RVC_GIT_URL",
    "https://github.com/RVC-Project/Retrieval-based-Voice-Conversion-WebUI.git",
)

# Mirror HF nếu huggingface.co hay bị reset (Trung Quốc / một số ISP)
os.environ.setdefault("RVC_HF_MIRROR", "0")

print("RVC_ROOT =", RVC_ROOT.resolve())
print("GIT_REPO =", GIT_REPO)


## 1) Cài đặt PyTorch GPU + thư viện

**Giải thích:** Colab thường có CUDA; ta cài `torch` bản CUDA và các gói RVC (`fairseq`, `ffmpeg-python`, …).  
`fairseq 0.12.2` hay lỗi resolver với `pip` quá mới — ta hạ `pip` vào khoảng 23.x–24.0.


In [ ]:
# --- CÀI PHỤ THUỘC ---
import subprocess
import sys

def sh(cmd: str) -> None:
    print(">>", cmd)
    r = subprocess.run(cmd, shell=True)
    if r.returncode != 0:
        raise SystemExit(r.returncode)

# Pip tương thích fairseq
sh(f"{sys.executable} -m pip install -q -U 'pip>=23.2,<24.1'")

# PyTorch: Colab/Kaggle — dùng index CUDA 12.x (Colab thường tương thích)
sh(f"{sys.executable} -m pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu124")

# RVC requirements cốt lõi (rút gọn; có thể mở rộng theo requirements.txt đầy đủ)
pkgs = [
    "numpy", "scipy", "librosa", "soundfile", "ffmpeg-python", "tensorboard",
    "tqdm", "faiss-cpu", "scikit-learn", "matplotlib", "dotenv",
    "fairseq==0.12.2", "requests", "praat-parselmouth", "pyworld",
]
sh(f"{sys.executable} -m pip install -q " + " ".join(pkgs))

# ffmpeg hệ thống (Colab thường có sẵn; nếu thiếu uncomment)
# sh("apt-get update -qq && apt-get install -qq ffmpeg")

import torch
print("torch", torch.__version__, "cuda?", torch.cuda.is_available())


## 2) Clone mã RVC (infer + configs)

**Giải thích:** Lệnh `git clone --depth 1` tải snapshot mỏng. Sau bước này, `RVC_ROOT` sẽ chứa `infer/`, `configs/`, … giống WebUI.


In [ ]:
# --- CLONE RVC ---
import os
import shutil
import subprocess
from pathlib import Path

def _as_path(x):
    return x if isinstance(x, Path) else Path(str(x))

root_dir = _as_path(RVC_ROOT)
if (root_dir / "infer").is_dir():
    print("Đã có infer/, bỏ qua clone:", root_dir)
else:
    parent = root_dir.parent
    parent.mkdir(parents=True, exist_ok=True)
    if root_dir.exists():
        shutil.rmtree(root_dir)
    subprocess.run(
        ["git", "clone", "--depth", "1", GIT_REPO, str(root_dir)],
        check=True,
    )
print("OK:", (root_dir / "infer").is_dir())


## 3) Ghi `training_pipeline` + các file đã patch (base64)

**Giải thích:** Các file được **mã hóa base64** lúc bạn chạy `python tools/generate_colab_notebook.py` trên máy. Ô code giải mã và ghi đè vào `RVC_ROOT` gồm:

- `training_pipeline/` — điều phối preprocess → F0 → Hubert → train → index  
- `infer/lib/fairseq_torch_load_compat.py` — PyTorch 2.6+ `torch.load`  
- `infer/modules/train/extract_feature_print.py` — `sys.path` + fairseq compat  
- `infer/modules/train/train.py` — `USE_LIBUV` trên Windows (và vẫn an toàn trên Linux)  
- `infer/lib/train/utils.py` — Matplotlib 3.8+ (`buffer_rgba`)  
- `infer/modules/vc/utils.py` — fairseq trước khi load Hubert (infer)  
- `tools/download_assets.py` — tải weight Hugging Face  

*Bản upstream RVC chỉ làm nền; các file trên đảm bảo lệnh train chạy trên môi trường mới.*


In [ ]:
from pathlib import Path
import base64
import os

try:
    _rv = RVC_ROOT
except NameError:
    _rv = os.environ.get('RVC_ROOT', '/content/Retrieval-based-Voice-Conversion-WebUI')
RVC_ROOT = Path(_rv)

# training_pipeline/__init__.py
_d = base64.b64decode("IyBUcmFpbmluZyBvcmNoZXN0cmF0aW9uIHBhY2thZ2UgKGLDqm4gdHJvbmcgcnZjX3N0YW5kYWxvbmUg4oCUIGtow7RuZyBwaOG7pSB0aHXhu5ljIHJlcG8gUlZDIGPFqSkuDQo=")
_p = RVC_ROOT / "training_pipeline/__init__.py"
_p.parent.mkdir(parents=True, exist_ok=True)
_p.write_bytes(_d)
print("Wrote", _p, "bytes", len(_d))

# training_pipeline/setup_env.py
_d = base64.b64decode("IiIiQ2jhu4kgZMO5bmcgYsOqbiB0cm9uZyB0aMawIG3hu6VjIHJ2Y19zdGFuZGFsb25lIOKAlCBn4buRYyA9IHRoxrAgbeG7pWMgY2hhIGPhu6dhIHBhY2thZ2UgdHJhaW5pbmdfcGlwZWxpbmUuIiIiDQpmcm9tIF9fZnV0dXJlX18gaW1wb3J0IGFubm90YXRpb25zDQoNCmltcG9ydCBvcw0KaW1wb3J0IHN5cw0KZnJvbSBwYXRobGliIGltcG9ydCBQYXRoDQoNCg0KZGVmIGdldF9zdGFuZGFsb25lX3Jvb3QoKSAtPiBQYXRoOg0KICAgIHJldHVybiBQYXRoKF9fZmlsZV9fKS5yZXNvbHZlKCkucGFyZW50LnBhcmVudA0KDQoNCmRlZiBib290c3RyYXAoY2xlYXJfYXJndl9mb3JfY29uZmlnOiBib29sID0gVHJ1ZSk6DQogICAgcm9vdCA9IGdldF9zdGFuZGFsb25lX3Jvb3QoKQ0KICAgIG9zLmNoZGlyKHJvb3QpDQogICAgcnMgPSBzdHIocm9vdCkNCiAgICBpZiBycyBub3QgaW4gc3lzLnBhdGg6DQogICAgICAgIHN5cy5wYXRoLmluc2VydCgwLCBycykNCg0KICAgIHRyeToNCiAgICAgICAgZnJvbSBkb3RlbnYgaW1wb3J0IGxvYWRfZG90ZW52DQoNCiAgICAgICAgbG9hZF9kb3RlbnYocm9vdCAvICIuZW52IikNCiAgICBleGNlcHQgSW1wb3J0RXJyb3I6DQogICAgICAgIHBhc3MNCg0KICAgIGZvciBkIGluIHR1cGxlKA0KICAgICAgICByb290IC8geA0KICAgICAgICBmb3IgeCBpbiAoDQogICAgICAgICAgICAiYXNzZXRzL3dlaWdodHMiLA0KICAgICAgICAgICAgImFzc2V0cy9pbmRpY2VzIiwNCiAgICAgICAgICAgICJhc3NldHMvaHViZXJ0IiwNCiAgICAgICAgICAgICJhc3NldHMvcHJldHJhaW5lZCIsDQogICAgICAgICAgICAiYXNzZXRzL3ByZXRyYWluZWRfdjIiLA0KICAgICAgICAgICAgImFzc2V0cy91dnI1X3dlaWdodHMiLA0KICAgICAgICAgICAgImFzc2V0cy9ybXZwZSIsDQogICAgICAgICAgICAibG9ncyIsDQogICAgICAgICAgICAiZGF0YXNldHMiLA0KICAgICAgICAgICAgIlRFTVAiLA0KICAgICAgICApDQogICAgKToNCiAgICAgICAgZC5ta2RpcihwYXJlbnRzPVRydWUsIGV4aXN0X29rPVRydWUpDQoNCiAgICBzYXZlZCA9IHN5cy5hcmd2WzpdDQogICAgaWYgY2xlYXJfYXJndl9mb3JfY29uZmlnOg0KICAgICAgICBzeXMuYXJndiA9IFtzYXZlZFswXSBpZiBzYXZlZCBlbHNlICJweXRob24iXQ0KICAgIHRyeToNCiAgICAgICAgZnJvbSBjb25maWdzLmNvbmZpZyBpbXBvcnQgQ29uZmlnDQoNCiAgICAgICAgcmV0dXJuIHJvb3QsIENvbmZpZygpDQogICAgZmluYWxseToNCiAgICAgICAgc3lzLmFyZ3YgPSBzYXZlZA0K")
_p = RVC_ROOT / "training_pipeline/setup_env.py"
_p.parent.mkdir(parents=True, exist_ok=True)
_p.write_bytes(_d)
print("Wrote", _p, "bytes", len(_d))

# training_pipeline/params.py
_d = base64.b64decode("IiIiDQpIeXBlcnBhcmFtZXRlcnMgZm9yIHRyYWluaW5nIChtaXJyb3IgV2ViVUkgIlRyYWluIiB0YWIpLg0KIiIiDQpmcm9tIF9fZnV0dXJlX18gaW1wb3J0IGFubm90YXRpb25zDQoNCmZyb20gZGF0YWNsYXNzZXMgaW1wb3J0IGRhdGFjbGFzcywgZmllbGQNCmZyb20gdHlwaW5nIGltcG9ydCBMaXRlcmFsDQoNCg0KU2FtcGxlUmF0ZUxhYmVsID0gTGl0ZXJhbFsiMzJrIiwgIjQwayIsICI0OGsiXQ0KUnZjVmVyc2lvbiA9IExpdGVyYWxbInYxIiwgInYyIl0NCg0KDQpAZGF0YWNsYXNzDQpjbGFzcyBUcmFpbmluZ1BhcmFtczoNCiAgICBleHBlcmltZW50X25hbWU6IHN0ciA9ICJteV92b2ljZSINCiAgICB0cmFpbnNldF9kaXI6IHN0ciA9ICJkYXRhc2V0cy9teV92b2ljZV93YXZzIg0KICAgIHNhbXBsZV9yYXRlX2xhYmVsOiBTYW1wbGVSYXRlTGFiZWwgPSAiNDBrIg0KICAgIHZlcnNpb246IFJ2Y1ZlcnNpb24gPSAidjIiDQogICAgaWZfZjA6IGJvb2wgPSBUcnVlDQogICAgc3BlYWtlcl9pZDogaW50ID0gMA0KICAgIG51bV9wcm9jZXNzZXM6IGludCA9IDQNCiAgICBmMF9tZXRob2Q6IHN0ciA9ICJybXZwZSINCiAgICBncHVzX2Zvcl9ybXZwZTogc3RyID0gIjAiDQogICAgZ3B1X2RldmljZXNfdHJhaW46IHN0ciA9ICIwIg0KICAgIHNhdmVfZXZlcnlfZXBvY2g6IGludCA9IDUNCiAgICB0b3RhbF9lcG9jaHM6IGludCA9IDIwMA0KICAgIGJhdGNoX3NpemU6IGludCA9IDQNCiAgICBzYXZlX29ubHlfbGF0ZXN0OiBib29sID0gVHJ1ZQ0KICAgIGNhY2hlX2RhdGFzZXRfaW5fZ3B1OiBib29sID0gRmFsc2UNCiAgICBzYXZlX3dlaWdodHNfZXZlcnlfZXBvY2g6IGJvb2wgPSBGYWxzZQ0KICAgIHByZXRyYWluZWRfZzogc3RyID0gIiINCiAgICBwcmV0cmFpbmVkX2Q6IHN0ciA9ICIiDQogICAgc2tpcF9pbmRleDogYm9vbCA9IEZhbHNlDQogICAgZXh0cmFjdF9pbmZlcl9wdGg6IGJvb2wgPSBGYWxzZQ0KICAgIGluZmVyX3dlaWdodF9uYW1lOiBzdHIgPSAibXlfdm9pY2VfaW5mZXIiDQogICAgZ19jaGVja3BvaW50X2Zvcl9leHRyYWN0OiBzdHIgPSAiIg0KICAgIGV4dHJhY3RfaW5mb19zdHI6IHN0ciA9ICJFeHRyYWN0ZWQgbW9kZWwuIg0KICAgIHNyX2RpY3Q6IGRpY3QgPSBmaWVsZCgNCiAgICAgICAgZGVmYXVsdF9mYWN0b3J5PWxhbWJkYTogeyIzMmsiOiAzMjAwMCwgIjQwayI6IDQwMDAwLCAiNDhrIjogNDgwMDB9DQogICAgKQ0KDQoNCmRlZiByZXNvbHZlX3ByZXRyYWluZWRfcGF0aHMocDogVHJhaW5pbmdQYXJhbXMsIHBhdGhfc3VmZml4X3YyOiBzdHIpIC0+IHR1cGxlW3N0ciwgc3RyXToNCiAgICBpbXBvcnQgb3MNCg0KICAgIGYwX3N0ciA9ICJmMCIgaWYgcC5pZl9mMCBlbHNlICIiDQogICAgc3IgPSBwLnNhbXBsZV9yYXRlX2xhYmVsDQoNCiAgICBkZWYgcHRoKGtpbmQ6IHN0cikgLT4gc3RyOg0KICAgICAgICByZXR1cm4gZiJhc3NldHMvcHJldHJhaW5lZHtwYXRoX3N1ZmZpeF92Mn0ve2YwX3N0cn17a2luZH17c3J9LnB0aCINCg0KICAgIHBnLCBwZCA9IHB0aCgiRyIpLCBwdGgoIkQiKQ0KICAgIGcgPSBwLnByZXRyYWluZWRfZyBpZiBwLnByZXRyYWluZWRfZyBlbHNlIChwZyBpZiBvcy5hY2Nlc3MocGcsIG9zLkZfT0spIGVsc2UgIiIpDQogICAgZCA9IHAucHJldHJhaW5lZF9kIGlmIHAucHJldHJhaW5lZF9kIGVsc2UgKHBkIGlmIG9zLmFjY2VzcyhwZCwgb3MuRl9PSykgZWxzZSAiIikNCiAgICByZXR1cm4gZywgZA0K")
_p = RVC_ROOT / "training_pipeline/params.py"
_p.parent.mkdir(parents=True, exist_ok=True)
_p.write_bytes(_d)
print("Wrote", _p, "bytes", len(_d))

# training_pipeline/steps.py
_d = base64.b64decode("...")
_p = RVC_ROOT / "training_pipeline/steps.py"
_p.parent.mkdir(parents=True, exist_ok=True)
_p.write_bytes(_d)
print("Wrote", _p, "bytes", len(_d))

# infer/lib/fairseq_torch_load_compat.py
_d = base64.b64decode("IiIiDQpQeVRvcmNoIDIuNisgZGVmYXVsdHMgdG9yY2gubG9hZCguLi4sIHdlaWdodHNfb25seT1UcnVlKS4NCkZhaXJzZXEgSHViZXJ0IGNoZWNrcG9pbnRzIHVucGlja2xlIGN1c3RvbSBjbGFzc2VzOyB0aGV5IG5lZWQgd2VpZ2h0c19vbmx5PUZhbHNlLg0KQXBwbHkgb25jZSBiZWZvcmUgYW55IGBpbXBvcnQgZmFpcnNlcWAgdGhhdCBsb2FkcyBodWJlcnRfYmFzZS5wdC4NCiIiIg0KZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucw0KDQppbXBvcnQgaW5zcGVjdA0KDQpfYXBwbGllZCA9IEZhbHNlDQoNCg0KZGVmIGFwcGx5X2ZhaXJzZXFfdG9yY2hfbG9hZF9jb21wYXQoKSAtPiBOb25lOg0KICAgIGdsb2JhbCBfYXBwbGllZA0KICAgIGlmIF9hcHBsaWVkOg0KICAgICAgICByZXR1cm4NCiAgICBpbXBvcnQgdG9yY2gNCg0KICAgIGlmICJ3ZWlnaHRzX29ubHkiIG5vdCBpbiBpbnNwZWN0LnNpZ25hdHVyZSh0b3JjaC5sb2FkKS5wYXJhbWV0ZXJzOg0KICAgICAgICBfYXBwbGllZCA9IFRydWUNCiAgICAgICAgcmV0dXJuDQogICAgX29yaWcgPSB0b3JjaC5sb2FkDQoNCiAgICBkZWYgX2xvYWQoKmFyZ3MsICoqa3dhcmdzKToNCiAgICAgICAga3dhcmdzLnNldGRlZmF1bHQoIndlaWdodHNfb25seSIsIEZhbHNlKQ0KICAgICAgICByZXR1cm4gX29yaWcoKmFyZ3MsICoqa3dhcmdzKQ0KDQogICAgdG9yY2gubG9hZCA9IF9sb2FkDQogICAgX2FwcGxpZWQgPSBUcnVlDQo=")
_p = RVC_ROOT / "infer/lib/fairseq_torch_load_compat.py"
_p.parent.mkdir(parents=True, exist_ok=True)
_p.write_bytes(_d)
print("Wrote", _p, "bytes", len(_d))

# infer/modules/train/extract_feature_print.py
_d = base64.b64decode("aW1wb3J0IG9zDQppbXBvcnQgc3lzDQppbXBvcnQgdHJhY2ViYWNrDQoNCiMgS2hpIGNo4bqheSBzdWJwcm9jZXNzOiBzeXMucGF0aFswXSBsw6AgdGjGsCBt4bulYyBjaOG7qWEgZmlsZSBuw6B5ICh0cmFpbi8pLCBraMO0bmcgcGjhuqNpIGfhu5FjIHJ2Y19zdGFuZGFsb25lLg0KX1JWQ19ST09UID0gb3MucGF0aC5hYnNwYXRoKG9zLnBhdGguam9pbihvcy5wYXRoLmRpcm5hbWUoX19maWxlX18pLCAiLi4iLCAiLi4iLCAiLi4iKSkNCmlmIF9SVkNfUk9PVCBub3QgaW4gc3lzLnBhdGg6DQogICAgc3lzLnBhdGguaW5zZXJ0KDAsIF9SVkNfUk9PVCkNCg0Kb3MuZW52aXJvblsiUFlUT1JDSF9FTkFCTEVfTVBTX0ZBTExCQUNLIl0gPSAiMSINCm9zLmVudmlyb25bIlBZVE9SQ0hfTVBTX0hJR0hfV0FURVJNQVJLX1JBVElPIl0gPSAiMC4wIg0KDQpkZXZpY2UgPSBzeXMuYXJndlsxXQ0Kbl9wYXJ0ID0gaW50KHN5cy5hcmd2WzJdKQ0KaV9wYXJ0ID0gaW50KHN5cy5hcmd2WzNdKQ0KaWYgbGVuKHN5cy5hcmd2KSA9PSA3Og0KICAgIGV4cF9kaXIgPSBzeXMuYXJndls0XQ0KICAgIHZlcnNpb24gPSBzeXMuYXJndls1XQ0KICAgIGlzX2hhbGYgPSBzeXMuYXJndls2XS5sb3dlcigpID09ICJ0cnVlIg0KZWxzZToNCiAgICBpX2dwdSA9IHN5cy5hcmd2WzRdDQogICAgZXhwX2RpciA9IHN5cy5hcmd2WzVdDQogICAgb3MuZW52aXJvblsiQ1VEQV9WSVNJQkxFX0RFVklDRVMiXSA9IHN0cihpX2dwdSkNCiAgICB2ZXJzaW9uID0gc3lzLmFyZ3ZbNl0NCiAgICBpc19oYWxmID0gc3lzLmFyZ3ZbN10ubG93ZXIoKSA9PSAidHJ1ZSINCmZyb20gaW5mZXIubGliLmZhaXJzZXFfdG9yY2hfbG9hZF9jb21wYXQgaW1wb3J0IGFwcGx5X2ZhaXJzZXFfdG9yY2hfbG9hZF9jb21wYXQNCg0KYXBwbHlfZmFpcnNlcV90b3JjaF9sb2FkX2NvbXBhdCgpDQoNCmltcG9ydCBmYWlyc2VxDQppbXBvcnQgbnVtcHkgYXMgbnANCmltcG9ydCBzb3VuZGZpbGUgYXMgc2YNCmltcG9ydCB0b3JjaA0KaW1wb3J0IHRvcmNoLm5uLmZ1bmN0aW9uYWwgYXMgRg0KDQppZiAicHJpdmF0ZXVzZW9uZSIgbm90IGluIGRldmljZToNCiAgICBkZXZpY2UgPSAiY3B1Ig0KICAgIGlmIHRvcmNoLmN1ZGEuaXNfYXZhaWxhYmxlKCk6DQogICAgICAgIGRldmljZSA9ICJjdWRhIg0KICAgIGVsaWYgdG9yY2guYmFja2VuZHMubXBzLmlzX2F2YWlsYWJsZSgpOg0KICAgICAgICBkZXZpY2UgPSAibXBzIg0KZWxzZToNCiAgICBpbXBvcnQgdG9yY2hfZGlyZWN0bWwNCg0KICAgIGRldmljZSA9IHRvcmNoX2RpcmVjdG1sLmRldmljZSh0b3JjaF9kaXJlY3RtbC5kZWZhdWx0X2RldmljZSgpKQ0KDQogICAgZGVmIGZvcndhcmRfZG1sKGN0eCwgeCwgc2NhbGUpOg0KICAgICAgICBjdHguc2NhbGUgPSBzY2FsZQ0KICAgICAgICByZXMgPSB4LmNsb25lKCkuZGV0YWNoKCkNCiAgICAgICAgcmV0dXJuIHJlcw0KDQogICAgZmFpcnNlcS5tb2R1bGVzLmdyYWRfbXVsdGlwbHkuR3JhZE11bHRpcGx5LmZvcndhcmQgPSBmb3J3YXJkX2RtbA0KDQpmID0gb3BlbigiJXMvZXh0cmFjdF9mMF9mZWF0dXJlLmxvZyIgJSBleHBfZGlyLCAiYSsiKQ0KDQoNCmRlZiBwcmludHQoc3Rycik6DQogICAgcHJpbnQoc3RycikNCiAgICBmLndyaXRlKCIlc1xuIiAlIHN0cnIpDQogICAgZi5mbHVzaCgpDQoNCg0KcHJpbnR0KCIgIi5qb2luKHN5cy5hcmd2KSkNCm1vZGVsX3BhdGggPSAiYXNzZXRzL2h1YmVydC9odWJlcnRfYmFzZS5wdCINCg0KcHJpbnR0KCJleHBfZGlyOiAiICsgZXhwX2RpcikNCndhdlBhdGggPSAiJXMvMV8xNmtfd2F2cyIgJSBleHBfZGlyDQpvdXRQYXRoID0gKA0KICAgICIlcy8zX2ZlYXR1cmUyNTYiICUgZXhwX2RpciBpZiB2ZXJzaW9uID09ICJ2MSIgZWxzZSAiJXMvM19mZWF0dXJlNzY4IiAlIGV4cF9kaXINCikNCm9zLm1ha2VkaXJzKG91dFBhdGgsIGV4aXN0X29rPVRydWUpDQoNCg0KIyB3YXZlIG11c3QgYmUgMTZrLCBob3Bfc2l6ZT0zMjANCmRlZiByZWFkd2F2ZSh3YXZfcGF0aCwgbm9ybWFsaXplPUZhbHNlKToNCiAgICB3YXYsIHNyID0gc2YucmVhZCh3YXZfcGF0aCkNCiAgICBhc3NlcnQgc3IgPT0gMTYwMDANCiAgICBmZWF0cyA9IHRvcmNoLmZyb21fbnVtcHkod2F2KS5mbG9hdCgpDQogICAgaWYgZmVhdHMuZGltKCkgPT0gMjogICMgZG91YmxlIGNoYW5uZWxzDQogICAgICAgIGZlYXRzID0gZmVhdHMubWVhbigtMSkNCiAgICBhc3NlcnQgZmVhdHMuZGltKCkgPT0gMSwgZmVhdHMuZGltKCkNCiAgICBpZiBub3JtYWxpemU6DQogICAgICAgIHdpdGggdG9yY2gubm9fZ3JhZCgpOg0KICAgICAgICAgICAgZmVhdHMgPSBGLmxheWVyX25vcm0oZmVhdHMsIGZlYXRzLnNoYXBlKQ0KICAgIGZlYXRzID0gZmVhdHMudmlldygxLCAtMSkNCiAgICByZXR1cm4gZmVhdHMNCg0KDQojIEh1QkVSVCBtb2RlbA0KcHJpbnR0KCJsb2FkIG1vZGVsKHMpIGZyb20ge30iLmZvcm1hdChtb2RlbF9wYXRoKSkNCiMgaWYgaHViZXJ0IG1vZGVsIGlzIGV4aXN0DQppZiBvcy5hY2Nlc3MobW9kZWxfcGF0aCwgb3MuRl9PSykgPT0gRmFsc2U6DQogICAgcHJpbnR0KA0KICAgICAgICAiRXJyb3I6IEV4dHJhY3RpbmcgaXMgc2h1dCBkb3duIGJlY2F1c2UgJXMgZG9lcyBub3QgZXhpc3QsIHlvdSBtYXkgZG93bmxvYWQgaXQgZnJvbSBodHRwczovL2h1Z2dpbmdmYWNlLmNvL2xqMTk5NS9Wb2ljZUNvbnZlcnNpb25XZWJVSS90cmVlL21haW4iDQogICAgICAgICUgbW9kZWxfcGF0aA0KICAgICkNCiAgICBleGl0KDApDQp0cnk6DQogICAgbW9kZWxzLCBzYXZlZF9jZmcsIHRhc2sgPSBmYWlyc2VxLmNoZWNrcG9pbnRfdXRpbHMubG9hZF9tb2RlbF9lbnNlbWJsZV9hbmRfdGFzaygNCiAgICAgICAgW21vZGVsX3BhdGhdLA0KICAgICAgICBzdWZmaXg9IiIsDQogICAgKQ0KICAgIG1vZGVsID0gbW9kZWxzWzBdDQogICAgbW9kZWwgPSBtb2RlbC50byhkZXZpY2UpDQogICAgcHJpbnR0KCJtb3ZlIG1vZGVsIHRvICVzIiAlIGRldmljZSkNCmV4Y2VwdCBFeGNlcHRpb246DQogICAgcHJpbnR0KHRyYWNlYmFjay5mb3JtYXRfZXhjKCkpDQogICAgcmFpc2UNCmlmIGlzX2hhbGY6DQogICAgaWYgZGV2aWNlIG5vdCBpbiBbIm1wcyIsICJjcHUiXToNCiAgICAgICAgbW9kZWwgPSBtb2RlbC5oYWxmKCkNCm1vZGVsLmV2YWwoKQ0KDQp0b2RvID0gc29ydGVkKGxpc3Qob3MubGlzdGRpcih3YXZQYXRoKSkpW2lfcGFydDo6bl9wYXJ0XQ0KbiA9IG1heCgxLCBsZW4odG9kbykgLy8gMTApICAjIOacgOWkmuaJk+WNsOWNgeadoQ0KaWYgbGVuKHRvZG8pID09IDA6DQogICAgcHJpbnR0KCJuby1mZWF0dXJlLXRvZG8iKQ0KZWxzZToNCiAgICBwcmludHQoImFsbC1mZWF0dXJlLSVzIiAlIGxlbih0b2RvKSkNCiAgICBmb3IgaWR4LCBmaWxlIGluIGVudW1lcmF0ZSh0b2RvKToNCiAgICAgICAgdHJ5Og0KICAgICAgICAgICAgaWYgZmlsZS5lbmRzd2l0aCgiLndhdiIpOg0KICAgICAgICAgICAgICAgIHdhdl9wYXRoID0gIiVzLyVzIiAlICh3YXZQYXRoLCBmaWxlKQ0KICAgICAgICAgICAgICAgIG91dF9wYXRoID0gIiVzLyVzIiAlIChvdXRQYXRoLCBmaWxlLnJlcGxhY2UoIndhdiIsICJucHkiKSkNCg0KICAgICAgICAgICAgICAgIGlmIG9zLnBhdGguZXhpc3RzKG91dF9wYXRoKToNCiAgICAgICAgICAgICAgICAgICAgY29udGludWUNCg0KICAgICAgICAgICAgICAgIGZlYXRzID0gcmVhZHdhdmUod2F2X3BhdGgsIG5vcm1hbGl6ZT1zYXZlZF9jZmcudGFzay5ub3JtYWxpemUpDQogICAgICAgICAgICAgICAgcGFkZGluZ19tYXNrID0gdG9yY2guQm9vbFRlbnNvcihmZWF0cy5zaGFwZSkuZmlsbF8oRmFsc2UpDQogICAgICAgICAgICAgICAgaW5wdXRzID0gew0KICAgICAgICAgICAgICAgICAgICAic291cmNlIjogKA0KICAgICAgICAgICAgICAgICAgICAgICAgZmVhdHMuaGFsZigpLnRvKGRldmljZSkNCiAgICAgICAgICAgICAgICAgICAgICAgIGlmIGlzX2hhbGYgYW5kIGRldmljZSBub3QgaW4gWyJtcHMiLCAiY3B1Il0NCiAgICAgICAgICAgICAgICAgICAgICAgIGVsc2UgZmVhdHMudG8oZGV2aWNlKQ0KICAgICAgICAgICAgICAgICAgICApLA0KICAgICAgICAgICAgICAgICAgICAicGFkZGluZ19tYXNrIjogcGFkZGluZ19tYXNrLnRvKGRldmljZSksDQogICAgICAgICAgICAgICAgICAgICJvdXRwdXRfbGF5ZXIiOiA5IGlmIHZlcnNpb24gPT0gInYxIiBlbHNlIDEyLCAgIyBsYXllciA5DQogICAgICAgICAgICAgICAgfQ0KICAgICAgICAgICAgICAgIHdpdGggdG9yY2gubm9fZ3JhZCgpOg0KICAgICAgICAgICAgICAgICAgICBsb2dpdHMgPSBtb2RlbC5leHRyYWN0X2ZlYXR1cmVzKCoqaW5wdXRzKQ0KICAgICAgICAgICAgICAgICAgICBmZWF0cyA9ICgNCiAgICAgICAgICAgICAgICAgICAgICAgIG1vZGVsLmZpbmFsX3Byb2oobG9naXRzWzBdKSBpZiB2ZXJzaW9uID09ICJ2MSIgZWxzZSBsb2dpdHNbMF0NCiAgICAgICAgICAgICAgICAgICAgKQ0KDQogICAgICAgICAgICAgICAgZmVhdHMgPSBmZWF0cy5zcXVlZXplKDApLmZsb2F0KCkuY3B1KCkubnVtcHkoKQ0KICAgICAgICAgICAgICAgIGlmIG5wLmlzbmFuKGZlYXRzKS5zdW0oKSA9PSAwOg0KICAgICAgICAgICAgICAgICAgICBucC5zYXZlKG91dF9wYXRoLCBmZWF0cywgYWxsb3dfcGlja2xlPUZhbHNlKQ0KICAgICAgICAgICAgICAgIGVsc2U6DQogICAgICAgICAgICAgICAgICAgIHByaW50dCgiJXMtY29udGFpbnMgbmFuIiAlIGZpbGUpDQogICAgICAgICAgICAgICAgaWYgaWR4ICUgbiA9PSAwOg0KICAgICAgICAgICAgICAgICAgICBwcmludHQoIm5vdy0lcyxhbGwtJXMsJXMsJXMiICUgKGxlbih0b2RvKSwgaWR4LCBmaWxlLCBmZWF0cy5zaGFwZSkpDQogICAgICAgIGV4Y2VwdDoNCiAgICAgICAgICAgIHByaW50dCh0cmFjZWJhY2suZm9ybWF0X2V4YygpKQ0KICAgIHByaW50dCgiYWxsLWZlYXR1cmUtZG9uZSIpDQo=")
_p = RVC_ROOT / "infer/modules/train/extract_feature_print.py"
_p.parent.mkdir(parents=True, exist_ok=True)
_p.write_bytes(_d)
print("Wrote", _p, "bytes", len(_d))

# infer/modules/train/train.py
_d = base64.b64decode("...")
_p = RVC_ROOT / "infer/modules/train/train.py"
_p.parent.mkdir(parents=True, exist_ok=True)
_p.write_bytes(_d)
print("Wrote", _p, "bytes", len(_d))

# infer/lib/train/utils.py
_d = base64.b64decode("...")
_p = RVC_ROOT / "infer/lib/train/utils.py"
_p.parent.mkdir(parents=True, exist_ok=True)
_p.write_bytes(_d)
print("Wrote", _p, "bytes", len(_d))

# infer/modules/vc/utils.py
_d = base64.b64decode("aW1wb3J0IG9zDQoNCmZyb20gaW5mZXIubGliLmZhaXJzZXFfdG9yY2hfbG9hZF9jb21wYXQgaW1wb3J0IGFwcGx5X2ZhaXJzZXFfdG9yY2hfbG9hZF9jb21wYXQNCg0KYXBwbHlfZmFpcnNlcV90b3JjaF9sb2FkX2NvbXBhdCgpDQoNCmZyb20gZmFpcnNlcSBpbXBvcnQgY2hlY2twb2ludF91dGlscw0KDQoNCmRlZiBnZXRfaW5kZXhfcGF0aF9mcm9tX21vZGVsKHNpZCk6DQogICAgcmV0dXJuIG5leHQoDQogICAgICAgICgNCiAgICAgICAgICAgIGYNCiAgICAgICAgICAgIGZvciBmIGluIFsNCiAgICAgICAgICAgICAgICBvcy5wYXRoLmpvaW4ocm9vdCwgbmFtZSkNCiAgICAgICAgICAgICAgICBmb3Igcm9vdCwgXywgZmlsZXMgaW4gb3Mud2Fsayhvcy5nZXRlbnYoImluZGV4X3Jvb3QiKSwgdG9wZG93bj1GYWxzZSkNCiAgICAgICAgICAgICAgICBmb3IgbmFtZSBpbiBmaWxlcw0KICAgICAgICAgICAgICAgIGlmIG5hbWUuZW5kc3dpdGgoIi5pbmRleCIpIGFuZCAidHJhaW5lZCIgbm90IGluIG5hbWUNCiAgICAgICAgICAgIF0NCiAgICAgICAgICAgIGlmIHNpZC5zcGxpdCgiLiIpWzBdIGluIGYNCiAgICAgICAgKSwNCiAgICAgICAgIiIsDQogICAgKQ0KDQoNCmRlZiBsb2FkX2h1YmVydChjb25maWcpOg0KICAgIG1vZGVscywgXywgXyA9IGNoZWNrcG9pbnRfdXRpbHMubG9hZF9tb2RlbF9lbnNlbWJsZV9hbmRfdGFzaygNCiAgICAgICAgWyJhc3NldHMvaHViZXJ0L2h1YmVydF9iYXNlLnB0Il0sDQogICAgICAgIHN1ZmZpeD0iIiwNCiAgICApDQogICAgaHViZXJ0X21vZGVsID0gbW9kZWxzWzBdDQogICAgaHViZXJ0X21vZGVsID0gaHViZXJ0X21vZGVsLnRvKGNvbmZpZy5kZXZpY2UpDQogICAgaWYgY29uZmlnLmlzX2hhbGY6DQogICAgICAgIGh1YmVydF9tb2RlbCA9IGh1YmVydF9tb2RlbC5oYWxmKCkNCiAgICBlbHNlOg0KICAgICAgICBodWJlcnRfbW9kZWwgPSBodWJlcnRfbW9kZWwuZmxvYXQoKQ0KICAgIHJldHVybiBodWJlcnRfbW9kZWwuZXZhbCgpDQo=")
_p = RVC_ROOT / "infer/modules/vc/utils.py"
_p.parent.mkdir(parents=True, exist_ok=True)
_p.write_bytes(_d)
print("Wrote", _p, "bytes", len(_d))

# tools/download_assets.py
_d = base64.b64decode("IiIiDQpU4bqjaSBwcmV0cmFpbmVkIC8gSHViZXJ0IC8gUk1WUEUgLyBVVlI1IHbDoG8gYXNzZXRzLyBj4bunYSBnw7NpIHJ2Y19zdGFuZGFsb25lLg0KDQpDaOG6oXkgdOG7qyBi4bqldCBr4buzIMSRw6J1Og0KICBjZCAvcGF0aC90by9ydmNfc3RhbmRhbG9uZQ0KICBweXRob24gdG9vbHMvZG93bmxvYWRfYXNzZXRzLnB5DQoNCkPhuqduOiBwaXAgaW5zdGFsbCByZXF1ZXN0cw0KDQpM4buXaSBDb25uZWN0aW9uUmVzZXRFcnJvciAoV2luRXJyb3IgMTAwNTQpIC8gQ29ubmVjdGlvbiBhYm9ydGVkOg0KICAtIE3huqFuZyAvIGZpcmV3YWxsIC8gVlBOIC8gbmjDoCBt4bqhbmcgxJHDs25nIGvhur90IG7hu5FpIEhUVFBTIHThu5tpIEh1Z2dpbmcgRmFjZS4NCiAgLSBDaOG6oXkgbOG6oWkgc2NyaXB0ICjEkcOjIGPDsyByZXRyeSB04buxIMSR4buZbmcpOyB0aOG7rSBWUE4g4buVbiDEkeG7i25oIGhv4bq3YyBt4bqhbmcga2jDoWMuDQogIC0gVGjhu60gbWlycm9yICht4buZdCBz4buRIHbDuW5nIG3huqFuZyDhu5VuIMSR4buLbmggaMahbik6DQogICAgICBzZXQgUlZDX0hGX01JUlJPUj0xDQogICAgICBweXRob24gdG9vbHMvZG93bmxvYWRfYXNzZXRzLnB5DQogICAgKFBvd2VyU2hlbGw6ICRlbnY6UlZDX0hGX01JUlJPUj0iMSI7IHB5dGhvbiB0b29scy9kb3dubG9hZF9hc3NldHMucHkpDQogIC0gSG/hurdjIHThuqNpIHRheTogaHR0cHM6Ly9odWdnaW5nZmFjZS5jby9sajE5OTUvVm9pY2VDb252ZXJzaW9uV2ViVUkvdHJlZS9tYWluDQoiIiINCmZyb20gX19mdXR1cmVfXyBpbXBvcnQgYW5ub3RhdGlvbnMNCg0KaW1wb3J0IG9zDQppbXBvcnQgdGltZQ0KZnJvbSBwYXRobGliIGltcG9ydCBQYXRoDQoNCmltcG9ydCByZXF1ZXN0cw0KDQojIFRoxrAgbeG7pWMgZ+G7kWMgPSBjaGEgY+G7p2EgdG9vbHMvIChjaMOtbmggbMOgIHJ2Y19zdGFuZGFsb25lKQ0KQkFTRV9ESVIgPSBQYXRoKF9fZmlsZV9fKS5yZXNvbHZlKCkucGFyZW50LnBhcmVudA0KDQpERUZBVUxUX0hGID0gImh0dHBzOi8vaHVnZ2luZ2ZhY2UuY28vbGoxOTk1L1ZvaWNlQ29udmVyc2lvbldlYlVJL3Jlc29sdmUvbWFpbi8iDQpNSVJST1JfSEYgPSAiaHR0cHM6Ly9oZi1taXJyb3IuY29tL2xqMTk5NS9Wb2ljZUNvbnZlcnNpb25XZWJVSS9yZXNvbHZlL21haW4vIg0KDQoNCmRlZiBfYmFzZV91cmwoKSAtPiBzdHI6DQogICAgaWYgb3MuZW52aXJvbi5nZXQoIlJWQ19IRl9CQVNFIik6DQogICAgICAgIHJldHVybiBvcy5lbnZpcm9uWyJSVkNfSEZfQkFTRSJdLnJzdHJpcCgiLyIpICsgIi8iDQogICAgaWYgb3MuZW52aXJvbi5nZXQoIlJWQ19IRl9NSVJST1IiLCAiIikubG93ZXIoKSBpbiAoIjEiLCAidHJ1ZSIsICJ5ZXMiKToNCiAgICAgICAgcmV0dXJuIE1JUlJPUl9IRg0KICAgIHJldHVybiBERUZBVUxUX0hGDQoNCg0KZGVmIF9zZXNzaW9uKCkgLT4gcmVxdWVzdHMuU2Vzc2lvbjoNCiAgICBzID0gcmVxdWVzdHMuU2Vzc2lvbigpDQogICAgcy5oZWFkZXJzLnVwZGF0ZSgNCiAgICAgICAgew0KICAgICAgICAgICAgIlVzZXItQWdlbnQiOiAiTW96aWxsYS81LjAgKGNvbXBhdGlibGU7IFJWQy1zdGFuZGFsb25lLWRvd25sb2FkLzEuMSkiLA0KICAgICAgICB9DQogICAgKQ0KICAgIHJldHVybiBzDQoNCg0KZGVmIGRsX21vZGVsKA0KICAgIGxpbms6IHN0ciwNCiAgICBtb2RlbF9uYW1lOiBzdHIsDQogICAgZGlyX25hbWU6IFBhdGgsDQogICAgKiwNCiAgICBzZXNzaW9uOiByZXF1ZXN0cy5TZXNzaW9uLA0KICAgIHJldHJpZXM6IGludCA9IDYsDQogICAgY29ubmVjdF90aW1lb3V0OiBpbnQgPSA2MCwNCiAgICByZWFkX3RpbWVvdXQ6IGludCA9IDYwMCwNCikgLT4gTm9uZToNCiAgICB1cmwgPSBmIntsaW5rfXttb2RlbF9uYW1lfSINCiAgICBvdXQgPSBkaXJfbmFtZSAvIG1vZGVsX25hbWUuc3BsaXQoIi8iKVstMV0NCiAgICBvdXQucGFyZW50Lm1rZGlyKHBhcmVudHM9VHJ1ZSwgZXhpc3Rfb2s9VHJ1ZSkNCiAgICB0aW1lb3V0ID0gKGNvbm5lY3RfdGltZW91dCwgcmVhZF90aW1lb3V0KQ0KICAgIGxhc3RfZXJyOiBFeGNlcHRpb24gfCBOb25lID0gTm9uZQ0KICAgIGZvciBhdHRlbXB0IGluIHJhbmdlKHJldHJpZXMpOg0KICAgICAgICB0bXA6IFBhdGggfCBOb25lID0gTm9uZQ0KICAgICAgICB0cnk6DQogICAgICAgICAgICB3aXRoIHNlc3Npb24uZ2V0KHVybCwgc3RyZWFtPVRydWUsIHRpbWVvdXQ9dGltZW91dCkgYXMgcjoNCiAgICAgICAgICAgICAgICByLnJhaXNlX2Zvcl9zdGF0dXMoKQ0KICAgICAgICAgICAgICAgIHRtcCA9IG91dC53aXRoX3N1ZmZpeChvdXQuc3VmZml4ICsgIi5wYXJ0IikNCiAgICAgICAgICAgICAgICB3aXRoIG9wZW4odG1wLCAid2IiKSBhcyBmOg0KICAgICAgICAgICAgICAgICAgICBmb3IgY2h1bmsgaW4gci5pdGVyX2NvbnRlbnQoY2h1bmtfc2l6ZT0xMDI0ICogMjU2KToNCiAgICAgICAgICAgICAgICAgICAgICAgIGlmIGNodW5rOg0KICAgICAgICAgICAgICAgICAgICAgICAgICAgIGYud3JpdGUoY2h1bmspDQogICAgICAgICAgICAgICAgdG1wLnJlcGxhY2Uob3V0KQ0KICAgICAgICAgICAgcmV0dXJuDQogICAgICAgIGV4Y2VwdCAocmVxdWVzdHMuUmVxdWVzdEV4Y2VwdGlvbiwgQ29ubmVjdGlvbkVycm9yLCBPU0Vycm9yKSBhcyBlOg0KICAgICAgICAgICAgbGFzdF9lcnIgPSBlDQogICAgICAgICAgICBpZiB0bXAgaXMgbm90IE5vbmUgYW5kIHRtcC5leGlzdHMoKToNCiAgICAgICAgICAgICAgICB0cnk6DQogICAgICAgICAgICAgICAgICAgIHRtcC51bmxpbmsoKQ0KICAgICAgICAgICAgICAgIGV4Y2VwdCBPU0Vycm9yOg0KICAgICAgICAgICAgICAgICAgICBwYXNzDQogICAgICAgICAgICB3YWl0ID0gbWluKDggKiAoMioqYXR0ZW1wdCksIDEyMCkNCiAgICAgICAgICAgIHByaW50KA0KICAgICAgICAgICAgICAgIGYiICBbIV0gTOG7l2kgdOG6o2kgKGzhuqduIHthdHRlbXB0ICsgMX0ve3JldHJpZXN9KToge2V9XG4iDQogICAgICAgICAgICAgICAgZiIgICAgICBDaOG7nSB7d2FpdH1zIHLhu5NpIHRo4butIGzhuqFpLi4uIg0KICAgICAgICAgICAgKQ0KICAgICAgICAgICAgdGltZS5zbGVlcCh3YWl0KQ0KICAgIHJhaXNlIFJ1bnRpbWVFcnJvcihmIlThuqNpIHRo4bqldCBi4bqhaSBzYXUge3JldHJpZXN9IGzhuqduOiB7dXJsfVxuR+G7kWMgbOG7l2k6IHtsYXN0X2Vycn0iKSBmcm9tIGxhc3RfZXJyDQoNCg0KaWYgX19uYW1lX18gPT0gIl9fbWFpbl9fIjoNCiAgICBiYXNlID0gX2Jhc2VfdXJsKCkNCiAgICBwcmludCgiQkFTRV9ESVIgPSIsIEJBU0VfRElSKQ0KICAgIHByaW50KCJIRiBiYXNlIFVSTCA9IiwgYmFzZSkNCiAgICBpZiBiYXNlID09IE1JUlJPUl9IRjoNCiAgICAgICAgcHJpbnQoIijEkGFuZyBkw7luZyBoZi1taXJyb3I7IMSR4bq3dCBSVkNfSEZfTUlSUk9SPTAgaG/hurdjIHjDs2EgYmnhur9uIMSR4buDIGTDuW5nIGh1Z2dpbmdmYWNlLmNvKSIpDQoNCiAgICBzZXNzID0gX3Nlc3Npb24oKQ0KDQogICAgcHJpbnQoIkRvd25sb2FkaW5nIGh1YmVydF9iYXNlLnB0Li4uIikNCiAgICBkbF9tb2RlbChiYXNlLCAiaHViZXJ0X2Jhc2UucHQiLCBCQVNFX0RJUiAvICJhc3NldHMvaHViZXJ0Iiwgc2Vzc2lvbj1zZXNzKQ0KICAgIHByaW50KCJEb3dubG9hZGluZyBybXZwZS5wdC4uLiIpDQogICAgZGxfbW9kZWwoYmFzZSwgInJtdnBlLnB0IiwgQkFTRV9ESVIgLyAiYXNzZXRzL3JtdnBlIiwgc2Vzc2lvbj1zZXNzKQ0KICAgIHByaW50KCJEb3dubG9hZGluZyB2b2NhbHMub25ueC4uLiIpDQogICAgZGxfbW9kZWwoDQogICAgICAgIGJhc2UgKyAidXZyNV93ZWlnaHRzL29ubnhfZGVyZXZlcmJfQnlfRm94Sm95LyIsDQogICAgICAgICJ2b2NhbHMub25ueCIsDQogICAgICAgIEJBU0VfRElSIC8gImFzc2V0cy91dnI1X3dlaWdodHMvb25ueF9kZXJldmVyYl9CeV9Gb3hKb3kiLA0KICAgICAgICBzZXNzaW9uPXNlc3MsDQogICAgKQ0KDQogICAgcnZjX21vZGVsc19kaXIgPSBCQVNFX0RJUiAvICJhc3NldHMvcHJldHJhaW5lZCINCiAgICBwcmludCgiRG93bmxvYWRpbmcgcHJldHJhaW5lZCBtb2RlbHMgKHYxIGZvbGRlcikuLi4iKQ0KICAgIG1vZGVsX25hbWVzID0gWw0KICAgICAgICAiRDMyay5wdGgiLA0KICAgICAgICAiRDQway5wdGgiLA0KICAgICAgICAiRDQ4ay5wdGgiLA0KICAgICAgICAiRzMyay5wdGgiLA0KICAgICAgICAiRzQway5wdGgiLA0KICAgICAgICAiRzQ4ay5wdGgiLA0KICAgICAgICAiZjBEMzJrLnB0aCIsDQogICAgICAgICJmMEQ0MGsucHRoIiwNCiAgICAgICAgImYwRDQ4ay5wdGgiLA0KICAgICAgICAiZjBHMzJrLnB0aCIsDQogICAgICAgICJmMEc0MGsucHRoIiwNCiAgICAgICAgImYwRzQ4ay5wdGgiLA0KICAgIF0NCiAgICBmb3IgbW9kZWwgaW4gbW9kZWxfbmFtZXM6DQogICAgICAgIHByaW50KGYiICB7bW9kZWx9Li4uIikNCiAgICAgICAgZGxfbW9kZWwoYmFzZSArICJwcmV0cmFpbmVkLyIsIG1vZGVsLCBydmNfbW9kZWxzX2Rpciwgc2Vzc2lvbj1zZXNzKQ0KDQogICAgcnZjX21vZGVsc19kaXIgPSBCQVNFX0RJUiAvICJhc3NldHMvcHJldHJhaW5lZF92MiINCiAgICBwcmludCgiRG93bmxvYWRpbmcgcHJldHJhaW5lZF92Mi4uLiIpDQogICAgZm9yIG1vZGVsIGluIG1vZGVsX25hbWVzOg0KICAgICAgICBwcmludChmIiAge21vZGVsfS4uLiIpDQogICAgICAgIGRsX21vZGVsKGJhc2UgKyAicHJldHJhaW5lZF92Mi8iLCBtb2RlbCwgcnZjX21vZGVsc19kaXIsIHNlc3Npb249c2VzcykNCg0KICAgIHJ2Y19tb2RlbHNfZGlyID0gQkFTRV9ESVIgLyAiYXNzZXRzL3V2cjVfd2VpZ2h0cyINCiAgICBwcmludCgiRG93bmxvYWRpbmcgdXZyNV93ZWlnaHRzLi4uIikNCiAgICB1dnJfbmFtZXMgPSBbDQogICAgICAgICJIUDItJUU0JUJBJUJBJUU1JUEzJUIwdm9jYWxzJTJCJUU5JTlEJTlFJUU0JUJBJUJBJUU1JUEzJUIwaW5zdHJ1bWVudGFscy5wdGgiLA0KICAgICAgICAiSFAyX2FsbF92b2NhbHMucHRoIiwNCiAgICAgICAgIkhQM19hbGxfdm9jYWxzLnB0aCIsDQogICAgICAgICJIUDUtJUU0JUI4JUJCJUU2JTk3JThCJUU1JUJFJThCJUU0JUJBJUJBJUU1JUEzJUIwdm9jYWxzJTJCJUU1JTg1JUI2JUU0JUJCJTk2aW5zdHJ1bWVudGFscy5wdGgiLA0KICAgICAgICAiSFA1X29ubHlfbWFpbl92b2NhbC5wdGgiLA0KICAgICAgICAiVlItRGVFY2hvQWdncmVzc2l2ZS5wdGgiLA0KICAgICAgICAiVlItRGVFY2hvRGVSZXZlcmIucHRoIiwNCiAgICAgICAgIlZSLURlRWNob05vcm1hbC5wdGgiLA0KICAgIF0NCiAgICBmb3IgbW9kZWwgaW4gdXZyX25hbWVzOg0KICAgICAgICBwcmludChmIiAge21vZGVsfS4uLiIpDQogICAgICAgIGRsX21vZGVsKGJhc2UgKyAidXZyNV93ZWlnaHRzLyIsIG1vZGVsLCBydmNfbW9kZWxzX2Rpciwgc2Vzc2lvbj1zZXNzKQ0KDQogICAgcHJpbnQoIkRvbmUuIEtp4buDbSB0cmEgYXNzZXRzL2h1YmVydCwgYXNzZXRzL3ByZXRyYWluZWQoX3YyKSwgYXNzZXRzL3JtdnBlLCBhc3NldHMvdXZyNV93ZWlnaHRzLiIpDQo=")
_p = RVC_ROOT / "tools/download_assets.py"
_p.parent.mkdir(parents=True, exist_ok=True)
_p.write_bytes(_d)
print("Wrote", _p, "bytes", len(_d))


## 5) Tải Hubert + pretrained (chạy từ `RVC_ROOT`)

**Giải thích:** Script `tools/download_assets.py` vừa được ghi ở bước 3. Ta `cd` vào `RVC_ROOT` rồi chạy. Có thể `export RVC_HF_MIRROR=1` nếu cần.


In [ ]:
# --- TẢI ASSETS ---
import os, subprocess, sys
from pathlib import Path

os.chdir(str(RVC_ROOT))
subprocess.run([sys.executable, "tools/download_assets.py"], check=True)


## 6) Thư mục `logs/mute`

**Giải thích:** Train cần template im lặng. Cách nhanh trên Colab: clone mute từ một bản có sẵn hoặc copy từ snapshot.

Nếu repo RVC của bạn **không** có `logs/mute`, hãy upload zip mute lên Colab hoặc tải từ bản WebUI đầy đủ.  
(Tùy chọn) Clone nhánh khác chỉ để lấy mute — có thể bỏ qua nếu đã có trong repo.


In [ ]:
# --- MUTE (tùy chọn — sửa URL nếu bạn có nguồn chứa logs/mute) ---
from pathlib import Path
import shutil, subprocess

m = Path(RVC_ROOT) / "logs" / "mute"
if (m / "0_gt_wavs").is_dir():
    print("mute OK")
else:
    print("THIEU logs/mute — vui long them tay (xem markdown o tren)")


## 7) Dataset — đặt file `.wav`

**Giải thích:** Upload qua panel Files của Colab vào `datasets/giong_cua_toi/` hoặc gắn Google Drive.


In [ ]:
# --- TẠO THƯ MỤC DATASET ---
from pathlib import Path
D = Path(RVC_ROOT) if not isinstance(RVC_ROOT, Path) else RVC_ROOT
(D / "datasets/giong_cua_toi").mkdir(parents=True, exist_ok=True)
print("Hay upload .wav vao:", D / "datasets/giong_cua_toi")


## 8) Train — bootstrap + từng bước

**Giải thích:**
- `bootstrap()` đưa `sys.path` và `cwd` về `RVC_ROOT`, nạp `configs.config.Config`.
- `TrainingParams` giống notebook để bàn: tên thí nghiệm, thư mục wav, epoch, batch, GPU.
- `train_steps.step_*` gọi subprocess `preprocess.py`, `extract_f0_print.py`, `extract_feature_print.py`, `train.py`, rồi FAISS index.


In [ ]:
# --- CHẠY TRAIN ---
import os, sys, logging, pathlib
from pathlib import Path

RVC_ROOT = Path(RVC_ROOT).resolve() if not isinstance(RVC_ROOT, Path) else RVC_ROOT.resolve()
os.chdir(str(RVC_ROOT))
sys.path.insert(0, str(RVC_ROOT))

logging.basicConfig(level=logging.INFO, format="%(levelname)s | %(message)s")

from training_pipeline.setup_env import bootstrap
from training_pipeline.params import TrainingParams
from training_pipeline import steps as train_steps

root, config = bootstrap()

p = TrainingParams(
    experiment_name="colab_voice",
    trainset_dir="datasets/giong_cua_toi",
    sample_rate_label="40k",
    version="v2",
    if_f0=True,
    num_processes=2,
    f0_method="rmvpe",
    gpu_devices_train="0",
    total_epochs=50,
    save_every_epoch=5,
    batch_size=4,
    skip_index=False,
)

miss = train_steps.check_mute_template(root)
print("logs/mute:", "THIEU" if miss else "OK", miss)

train_steps.step_preprocess(root, config, p)
train_steps.step_extract_f0_and_features(root, config, p)
train_steps.step_train(root, config, p)
if not p.skip_index:
    for line in train_steps.step_train_index(root, config, p):
        print(line)

print("Hoan tat pipeline (train + index). Checkpoint trong logs/", p.experiment_name)
